In [ ]:

# Setup & Imports
!pip install rdkit
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.model_selection import (
    RandomizedSearchCV, cross_validate, StratifiedKFold
)
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, roc_auc_score, matthews_corrcoef,
    average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve, auc,
    classification_report
)

print("Imports successful")

In [ ]:
# CELL 1: Load & Inspect Data
# Col 0 = SMILES, Col 1 = labels, Col 2+ = ECFP or chemical descriptors, fingerprints

TRAIN_CSV = '/content/train.csv'     #Chemical Fingerprints and descriptors etc are precalculated outside of this main training script for both train & test sets
TEST_CSV  = '/content/test.csv'

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

feature_cols = [col for col in train_df.columns if col not in ['smiles', 'labels']]

X_train = train_df[feature_cols]
y_train = train_df['labels']
X_test  = test_df[feature_cols]
y_test  = test_df['labels']

print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")
print(f"Fingerprint bits: {X_train.shape[1]}")
print(f"\nTrain class distribution:\n{y_train.value_counts()}")
print(f"\nTest class distribution:\n{y_test.value_counts()}")

In [ ]:
# CELL 2: Hyperparameter Tuning  (scored on MCC)
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

param_grid = dict(
    n_estimators=[100, 300, 500, 800],
    max_depth=[6, 10, 20, None],
    learning_rate=[0.05, 0.1, 0.2, 0.3],
    subsample=[0.8, 0.9, 1.0],
    colsample_bytree=[0.8, 0.9, 1.0],
    min_child_weight=[1, 5, 10],
    gamma=[0, 0.1, 0.2]
)

search = RandomizedSearchCV(
    XGBClassifier(
        scale_pos_weight=scale_pos,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    ),
    param_grid,
    n_iter=50,
    scoring='matthews_corrcoef',
    cv=5,
    random_state=42,
    n_jobs=-1,
    refit=True
)
search.fit(X_train, y_train)

print(f"Best hyperparameters : {search.best_params_}")
print(f"Best CV MCC (search) : {search.best_score_:.3f}")

In [ ]:
# CELL 3: Train Final XGB  +  5-Fold CV

cv_strategy     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_metrics = ['accuracy', 'f1', 'roc_auc','average_precision', 'precision', 'recall', 'matthews_corrcoef']

cv_results = cross_validate(
    XGBClassifier(
        scale_pos_weight=scale_pos,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1,
        **search.best_params_
    ),
    X_train, y_train,
    cv=cv_strategy,
    scoring=scoring_metrics
)

print("--- 5-Fold CV on Training Set ---")
for metric in scoring_metrics:
    m = cv_results[f'test_{metric}'].mean()
    s = cv_results[f'test_{metric}'].std()
    print(f"  {metric.replace('_', ' ').title():>22s}: {m:.3f} ± {s:.3f}")

model    = search.best_estimator_
y_pred   = model.predict(X_test)
y_proba  = model.predict_proba(X_test)[:, 1]

print("\nTraining complete")

In [ ]:
# CELL 4: Evaluation and Essential Metrics
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_proba)
pr_auc = auc(rec_curve, prec_curve)

metrics = {
    "Accuracy":           accuracy_score(y_test, y_pred),
    "Balanced Accuracy":  balanced_accuracy_score(y_test, y_pred),
    "F1-Score (macro)":   f1_score(y_test, y_pred, average='macro'),
    "F1-Score (class 1)": f1_score(y_test, y_pred, pos_label=1),
    "Sensitivity (TPR)":  tp / (tp + fn) if (tp + fn) > 0 else 0,
    "Specificity (TNR)":  tn / (tn + fp) if (tn + fp) > 0 else 0,
    "MCC":                matthews_corrcoef(y_test, y_pred),
    "ROC-AUC":            roc_auc_score(y_test, y_proba),
    "PR-AUC":             pr_auc,
    "TP": tp, "TN": tn, "FP": fp, "FN": fn
}

print("=" * 45)
print("   CLASSIFICATION RESULTS — XGBoost")
print("=" * 45)
for k, v in metrics.items():
    val = f"{v:.4f}" if isinstance(v, float) else str(v)
    print(f"  {k:<28}: {val}")

print("\n" + classification_report(y_test, y_pred,
      target_names=["Non-inhibitor (0)", "inhibitor (1)"], digits=4))

# Save metrics
pd.DataFrame([metrics]).T.rename(columns={0: "Value"}).to_csv('xgb_metrics.csv')
print("Saved: xgb_metrics.csv")

In [ ]:
# CELL 5: Visualizations — CM, ROC, PR Curves
fig = plt.figure(figsize=(18, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

#Confusion Matrix
ax0 = fig.add_subplot(gs[0])
im  = ax0.imshow(cm, cmap='Blues')
ax0.set_xticks([0, 1]); ax0.set_yticks([0, 1])
ax0.set_xticklabels(['Non-inhibitor', 'Inhibitor'])
ax0.set_yticklabels(['Non-inhibitor', 'Inhibitor'])
ax0.set_xlabel('Predicted'); ax0.set_ylabel('Actual')
ax0.set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        ax0.text(j, i, cm[i, j], ha='center', va='center',
                 fontsize=16, color='white' if cm[i, j] > cm.max()/2 else 'black')
fig.colorbar(im, ax=ax0)

#ROC Curve
ax1 = fig.add_subplot(gs[1])
fpr, tpr, _ = roc_curve(y_test, y_proba)
ax1.plot(fpr, tpr, 'steelblue', lw=2,
         label=f'AUC = {metrics["ROC-AUC"]:.3f}')
ax1.plot([0, 1], [0, 1], 'k--', lw=1)
ax1.fill_between(fpr, tpr, alpha=0.15, color='steelblue')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve')
ax1.legend(); ax1.grid(alpha=0.3)

#Precision-Recall Curve
ax2 = fig.add_subplot(gs[2])
baseline_pr = (y_test == 1).mean()
ax2.plot(rec_curve, prec_curve, 'tomato', lw=2,
         label=f'AUC = {pr_auc:.3f}')
ax2.axhline(baseline_pr, color='k', ls='--', lw=1,
            label=f'Baseline = {baseline_pr:.2f}')
ax2.fill_between(rec_curve, prec_curve, alpha=0.15, color='tomato')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('XGBoost Model Evaluation', fontsize=13, y=1.02)
plt.savefig('xgb_evaluation_plots.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: xgb_evaluation_plots.png")

In [ ]:
# CELL 6: Feature Importance  (Gini — Top 20 & Top 35)
importance_df = pd.DataFrame({
    'Feature':    feature_cols,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print("Top 35 Features (Gini Importance):")
print(importance_df.head(35).to_string(index=False))

# Top-20 bar chart
plt.figure(figsize=(10, 7))
top20 = importance_df.head(20)
plt.barh(top20['Feature'][::-1], top20['Importance'][::-1], color='steelblue')
plt.xlabel('Gini Importance')
plt.title('Top 20 Most Important Features — XGB')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('xgb_feature_importance_top20.png', dpi=150, bbox_inches='tight')
plt.show()

# Top-35 bar chart
fig, ax = plt.subplots(figsize=(9, 10))
top35 = importance_df.head(35)
ax.barh(top35['Feature'][::-1], top35['Importance'][::-1], color='steelblue')
ax.set_xlabel('Gini Importance')
ax.set_title('Top 35 Most Important Features — XgBoost')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('xgb_feature_importance_top35.png', dpi=150, bbox_inches='tight')
plt.show()

importance_df.to_csv('xgb_feature_importance.csv', index=False)
print("Saved: xgb_feature_importance_top20.png | xgb_feature_importance_top35.png | xgb_feature_importance.csv")

In [ ]:
# CELL 6.5: SHAP Feature Importance Analysis
import shap
import joblib
import matplotlib.pyplot as plt
import matplotlib as mpl

# SHAP Plotting Parameters
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
mpl.rcParams['font.size'] = 10
mpl.rcParams['axes.titlesize'] = 12
mpl.rcParams['axes.labelsize'] = 10
mpl.rcParams['xtick.labelsize'] = 9
mpl.rcParams['ytick.labelsize'] = 9
mpl.rcParams['axes.linewidth'] = 0.8
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['figure.dpi'] = 600


print("Loading saved model and calculating SHAP values...")
# 1 Load the pre-trained model
bundle_loaded = joblib.load('xgb_model.joblib')
loaded_model  = bundle_loaded['model']
feature_cols  = bundle_loaded['features']

shap.initjs()

# 2 Explain the loaded model's predictions
explainer = shap.TreeExplainer(loaded_model)
shap_values = explainer.shap_values(X_test)

# 3 SHAP Summary Plot
fig1 = plt.figure(figsize=(7, 5)) # Compact size for 1 or 2-column journal formats
shap.summary_plot(shap_values, X_test, feature_names=feature_cols, max_display=15, show=False)
plt.title("SHAP Feature Importance", pad=15, fontweight='bold')
plt.tight_layout()
plt.savefig('xgb_shap_summary_plot_nature.png', dpi=600, bbox_inches='tight')
plt.savefig('xgb_shap_summary_plot_nature.pdf', format='pdf', bbox_inches='tight')
plt.show()

# 4 SHAP Bar Plot
fig2 = plt.figure(figsize=(7, 5))
shap.summary_plot(shap_values, X_test, feature_names=feature_cols, plot_type="bar",
                  max_display=15, show=False, color="#4A90E2")
plt.title("Mean Absolute SHAP Value", pad=15, fontweight='bold')
plt.xlabel("Mean |SHAP value| (Average impact on model output)")
plt.tight_layout()
plt.savefig('xgb_shap_bar_plot_nature.png', dpi=600, bbox_inches='tight')
plt.savefig('xgb_shap_bar_plot_nature.pdf', format='pdf', bbox_inches='tight')
plt.show()

joblib.dump({'shap_values': shap_values, 'base_values': explainer.expected_value}, 'xgb_shap_data.joblib')

print(" plots Saved: .png and .pdf")

In [ ]:
# CELL 7: Save Model weights
joblib.dump({
    'model':       model,
    'features':    feature_cols,
    'best_params': search.best_params_
}, 'xgb_model.joblib')

# Save predictions with probabilities (for downstream analysis)
pd.DataFrame({
    'true_label':      y_test.values,
    'predicted_label': y_pred,
    'prob_nonihibitor':   model.predict_proba(X_test.values)[:, 0],
    'prob_inhibitor':      y_proba,
}).to_csv('rf_predictions.csv', index=False)

print("Model artifacts saved:")
print("   xgb_model.joblib      ← trained classifier + params")
print("   xgb_predictions.csv   ← test set predictions + probabilities")

In [ ]:
# CELL A: Build Applicability Domain from Training Set it Uses ECFP (radius=3, nBits=2048) consistent with the fingerprints used to train the model.
!pip install rdkit
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

AD_THRESHOLD = 0.5

def smiles_to_fp(smiles_series, radius=3, nbits=2048):
    fps = []
    for smi in smiles_series:
        mol = Chem.MolFromSmiles(smi)
        fps.append(
            AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits)
            if mol else None
        )
    return fps

print("Computing training set fingerprints...")
train_fps = smiles_to_fp(train_df['smiles'])
train_fps = [fp for fp in train_fps if fp is not None]

ad_bundle = {
    'train_fps': train_fps,
    'threshold': AD_THRESHOLD,
    'n_train':   len(train_fps),
    'radius':    3,
    'nbits':     2048,
}
joblib.dump(ad_bundle, 'xgb_ad_filter.joblib')

print(f" AD filter built from {len(train_fps)} training compounds")
print(f"   Tanimoto threshold : {AD_THRESHOLD}")
print(f"   ECFP radius        : 3  |  nBits: 2048")
print(f"   Saved: xgb_ad_filter.joblib")

In [ ]:
# CELL 8: Load Model & Screen New Compounds
# Usage: provide a CSV with SMILES col + 2048 ECFP bit columns (radius=3). Run ECFP generation on the chemical database first.
bundle_loaded = joblib.load('xgb_model.joblib')
model_loaded  = bundle_loaded['model']
feat_loaded   = bundle_loaded['features']

#Load compound library (post-AD filter, with ECFP features)
new_df     = pd.read_csv('file path')
smiles_col = new_df['smiles']
X_new      = new_df[feat_loaded].values


proba_new = model_loaded.predict_proba(X_new)[:, 1]

THRESHOLD = 0.50
TOP_N     = 50    # number of top hits to save separately

pred_new  = (proba_new >= THRESHOLD).astype(int)

hits = pd.DataFrame({
    'SMILES':          smiles_col.values,
    'Predicted_Class': pred_new,
    'Prob_Active':      proba_new
}).sort_values('Prob_Active', ascending=False)

hits_active = hits[hits['Predicted_Class'] == 1]

print(f"Total compounds screened   : {len(hits)}")
print(f"Predicted Active (≥{THRESHOLD})  : {len(hits_active)}")
print(f"\nTop {TOP_N} Predicted Hits:")
print(hits_active.head(TOP_N).to_string(index=False))

hits.to_csv('xgb_all_screened_compounds.csv', index=False)
hits_active.head(TOP_N).to_csv('xgb_top50_hits.csv', index=False)

print(f"\n Saved: xgb_all_screened_compounds.csv  ({len(hits)} compounds)")
print(f" Saved: xgb_top50_hits.csv               (top {TOP_N} predicted active)")

In [ ]:
# CELL 8b: Load Model & Screen New Compounds (Low-RAM Optimized) adapted to colab as it has low ram and crashes with large chemical database
# Usage: provide a Parquet file with SMILES col + 2048 ECFP bit columns (radius=3). we adapted it to avoid Colab crashes as it processes it in smaller chunks.
!pip install fastparquet pyarrow tqdm -q
import pandas as pd
import joblib
import pyarrow.parquet as pq
import gc
from tqdm.auto import tqdm

# Load Model
bundle_loaded = joblib.load('xgb_model.joblib')
model_loaded  = bundle_loaded['model']
feat_loaded   = bundle_loaded['features']

PARQUET_PATH = '/content/molportfull_ecfp_features.parquet' # Update filename if different
OUT_ALL      = 'xgb_all_screened_compounds.csv'
OUT_HITS     = 'xgb_top50_hits.csv'

THRESHOLD = 0.85
TOP_N     = 50
CHUNK_SIZE = 25_000

# Initialize Parquet Streaming
parquet_file = pq.ParquetFile(PARQUET_PATH)
total_rows = parquet_file.metadata.num_rows

print(f"Total rows to process: {total_rows:,}")
print(f"Streaming in chunks of {CHUNK_SIZE:,} to save RAM...")

first_chunk = True
active_hits_list = [] # Store only active hits in memory to sort later
total_screened = 0

#3. Process in Batches
with tqdm(total=total_rows, desc="Screening Compounds") as pbar:
    for batch in parquet_file.iter_batches(batch_size=CHUNK_SIZE):
        chunk = batch.to_pandas()

        smiles_batch = chunk['smiles'].values
        # Ensure columns are in the exact order the model expects
        X_batch = chunk[feat_loaded].values

        # Predict
        proba_batch = model_loaded.predict_proba(X_batch)[:, 1]
        pred_batch  = (proba_batch >= THRESHOLD).astype(int)

        # Create a lightweight results dataframe for this chunk
        chunk_results = pd.DataFrame({
            'SMILES': smiles_batch,
            'Predicted_Class': pred_batch,
            'Prob_Active': proba_batch
        })

        mode = 'w' if first_chunk else 'a'
        header = True if first_chunk else False
        chunk_results.to_csv(OUT_ALL, mode=mode, index=False, header=header)

        actives_chunk = chunk_results[chunk_results['Predicted_Class'] == 1]
        if not actives_chunk.empty:
            active_hits_list.append(actives_chunk)

        total_screened += len(chunk)
        first_chunk = False

        # Aggressive RAM cleanup
        del chunk, X_batch, smiles_batch, proba_batch, pred_batch, chunk_results, actives_chunk
        gc.collect()
        pbar.update(len(batch))

#Final Aggregation of Top Hits
print("Aggregating top hits")
if active_hits_list:
    # Combine all the >0.85 hits we found across all chunks
    all_actives = pd.concat(active_hits_list, ignore_index=True)
    all_actives = all_actives.sort_values('Prob_Active', ascending=False)

    hits_active = all_actives
    top_hits_df = all_actives.head(TOP_N)
else:
    hits_active = pd.DataFrame(columns=['SMILES', 'Predicted_Class', 'Prob_Active'])
    top_hits_df = hits_active

top_hits_df.to_csv(OUT_HITS, index=False)

#5Print Summary
print("=" * 45)
print(f"Total compounds screened   : {total_screened:,}")
print(f"Predicted Active (≥{THRESHOLD})  : {len(hits_active):,}")
print("=" * 45)

if not top_hits_df.empty:
    print(f"\nTop {TOP_N} Predicted Hits:")
    print(top_hits_df.to_string(index=False))

print(f"\nSaved: {OUT_ALL}  (All predictions)")
print(f"Saved: {OUT_HITS}              (Top {TOP_N} predicted active)")